In [3]:
import torch
import matplotlib.pyplot as plt
from src.unet import UNet
import ipywidgets as widgets
from IPython.display import display


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando motor: {device}")

# Instancia del modelo
unet = UNet(in_channels=3, out_channels=3).to(device)

# Cargar los pesos entrenados
unet.load_state_dict(torch.load("pokemon_unet.pth", weights_only=True, map_location=device))
unet.eval() 
print("¡Cerebro Pokémon cargado con éxito!")

def generar_pokemones_interactivo(n_samples, steps):
    print(f"Iniciando simulación matemática para {n_samples} Pokémon en {steps} pasos...")
    
    dt = 1.0 / steps
    z = torch.randn((n_samples, 3, 64, 64), device=device)
    
    historial_imagenes = []
    pasos_a_guardar = [0, int(steps*0.33), int(steps*0.66), steps - 1] 

    # 1. Generación Matemática
    with torch.no_grad():
        for step in range(steps):
            t_current = step * dt
            t_tensor = torch.full((n_samples,), t_current * 1000.0, device=device)
            x0_pred = unet(z, t_tensor)
            v = (x0_pred - z) / (1.0 - t_current + 1e-5)
            z = z + (v * dt)
            
            if step in pasos_a_guardar:
                img_visible = (z.clone().cpu().clamp(-1, 1) + 1) / 2
                historial_imagenes.append(img_visible)
                
    # 2. Ploteo Dinámico
    
    fig, axes = plt.subplots(4, n_samples, figsize=(n_samples * 2.5, 10))
    titulos_tiempo = ["Ruido (t=1.0)", "Sombras (t=0.66)", "Contornos (t=0.33)", "Final (t=0.0)"]

    # Protección matemática por si el usuario elige generar solo 1 Pokémon
    if n_samples == 1:
        axes = axes.reshape(4, 1)

    for idx_tiempo, lote_imagenes in enumerate(historial_imagenes):
        for idx_pokemon in range(n_samples):
            img_np = lote_imagenes[idx_pokemon].permute(1, 2, 0).numpy().clip(0, 1)
            ax = axes[idx_tiempo, idx_pokemon]
            ax.imshow(img_np)
            ax.axis("off")
            
            if idx_pokemon == 0:
                ax.set_title(titulos_tiempo[idx_tiempo], loc='left', fontweight='bold', fontsize=12)

    plt.tight_layout()
    plt.show()

# 3. CREACIÓN DE LA INTERFAZ DE USUARIO (UI)
interfaz = widgets.interact_manual(
    generar_pokemones_interactivo, 
    n_samples=widgets.IntSlider(min=1, max=8, step=1, value=4, description='Cantidad:'),
    steps=widgets.IntSlider(min=10, max=200, step=10, value=50, description='Pasos ODE:')
)

interfaz.widget.children[2].description = '¡Generar Pokémon!'
interfaz.widget.children[2].button_style = 'success' # Poner el botón verde


Usando motor: cuda
¡Cerebro Pokémon cargado con éxito!


interactive(children=(IntSlider(value=4, description='Cantidad:', max=8, min=1), IntSlider(value=50, descripti…